# Phase 0: Creating the GLYCO_GENES_WIDE Table

Extract glycogene expression data from the LINCS L1000 Chemical Perturbations (2021) dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import os
import logging
from typing import List, Optional, Dict
import gzip
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import serialization
from snowflake.connector import connect
from cmapPy.pandasGEXpress.parse import parse

# Logging configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

logger.info(f"Project root: {project_root}")

## Data File Path Configuration

Configure paths for LINCS data files and glycogene list.

In [ ]:
# Data file paths
data_dir = project_root / "sample" / "data"
glycogenes_list_path = project_root / "GlycoEnzOnto" / "glycogenes_from_gmt.txt"

# LINCS L1000 Chemical Perturbations (2021) data file
# Adjust file name to match actual downloaded file
# Common file name candidates:
# - l1000_cp.gctx
# - l1000_cp_n12327x720216.gctx
# - LINCS_L1000_Chemical_Perturbations_2021.gctx
gctx_file = data_dir / "l1000_cp.gctx"  # Change to match actual file name

# Metadata files (may be included in gctx file)
# Check and configure LINCS L1000 Chemical Perturbations (2021) metadata file names
pert_info_file = data_dir / "l1000_cp_pert_info.txt.gz"  # Change to match actual file name
sig_info_file = data_dir / "l1000_cp_sig_info.txt.gz"  # Change to match actual file name
gene_info_file = data_dir / "l1000_cp_gene_info.txt.gz"  # Change to match actual file name

# Check file existence
files_to_check = {
    "gctx": gctx_file,
    "glycogenes_list": glycogenes_list_path
}

# Metadata files are optional (may be included in gctx file)
optional_files = {
    "pert_info": pert_info_file,
    "sig_info": sig_info_file,
    "gene_info": gene_info_file
}

logger.info("Checking required files:")
for name, path in files_to_check.items():
    if path.exists():
        logger.info(f"✓ {name}: {path}")
    else:
        logger.error(f"✗ {name} not found: {path}")
        raise FileNotFoundError(f"{name} not found: {path}")

logger.info("\nChecking optional files:")
for name, path in optional_files.items():
    if path.exists():
        logger.info(f"✓ {name}: {path}")
    else:
        logger.warning(f"⚠ {name} not found: {path} (may be retrieved from gctx file)")

## Loading the Glycogene List

Load the glycogene list obtained from GlycoEnzOnto.

In [ ]:
# Load glycogene list
with open(glycogenes_list_path, 'r') as f:
    glycogenes_raw = [line.strip().strip('"') for line in f if line.strip()]

# Remove duplicates and sort
glycogenes = sorted(list(set(glycogenes_raw)))

logger.info(f"Number of glycogenes: {len(glycogenes)}")
logger.info(f"First 10 genes: {glycogenes[:10]}")
logger.info(f"Last 10 genes: {glycogenes[-10:]}")

## Loading LINCS Data

Load GSE92742 Level 5 data (gctx format) using the cmapPy library.

In [ ]:
logger.info("Loading LINCS L1000 Chemical Perturbations (2021) GCTX file...")
logger.info(f"File size: {gctx_file.stat().st_size / (1024**3):.2f} GB")
logger.info(f"Expected data dimensions: 12327 genes × 720216 samples")

# Load gctx file using cmapPy
gctoo = parse(str(gctx_file))

logger.info(f"Data shape: {gctoo.data_df.shape}")
logger.info(f"Number of rows (genes): {len(gctoo.data_df)}")
logger.info(f"Number of columns (samples): {len(gctoo.data_df.columns)}")

# Check metadata
if hasattr(gctoo, 'row_metadata_df') and gctoo.row_metadata_df is not None:
    logger.info(f"Row metadata (gene info): {gctoo.row_metadata_df.shape}")
    logger.info(f"Row metadata columns: {gctoo.row_metadata_df.columns.tolist()}")
else:
    logger.warning("Row metadata not found.")

if hasattr(gctoo, 'col_metadata_df') and gctoo.col_metadata_df is not None:
    logger.info(f"Column metadata (sample info): {gctoo.col_metadata_df.shape}")
    logger.info(f"Column metadata columns: {gctoo.col_metadata_df.columns.tolist()}")
else:
    logger.warning("Column metadata not found.")

# Check first few rows of data
logger.info("\nFirst 5 rows and 5 columns of data:")
logger.info(gctoo.data_df.iloc[:5, :5])

## Loading Gene Information and Extracting Glycogenes

Load gene information file and identify glycogene indices.

In [ ]:
# Load gene information (from gctx file row metadata or external file)
logger.info("Retrieving gene information...")

# First, check gctx file row metadata
if hasattr(gctoo, 'row_metadata_df') and gctoo.row_metadata_df is not None:
    if 'pr_gene_symbol' in gctoo.row_metadata_df.columns:
        logger.info("Retrieving gene information from gctx file row metadata.")
        gene_info_df = gctoo.row_metadata_df.copy()
        # Use index as gene ID
        gene_info_df['pr_gene_id'] = gene_info_df.index.astype(str)
    else:
        logger.warning("pr_gene_symbol not found in row metadata. Attempting to retrieve from external file.")
        gene_info_df = None
else:
    logger.info("Row metadata not found in gctx file. Retrieving from external file.")
    gene_info_df = None

# Load from external file (if needed)
if gene_info_df is None:
    if gene_info_file.exists():
        logger.info(f"Loading gene information file: {gene_info_file}")
        gene_info_df = pd.read_csv(gene_info_file, sep='\t', compression='gzip')
        logger.info(f"Gene information: {gene_info_df.shape}")
        logger.info(f"Columns: {gene_info_df.columns.tolist()}")
    else:
        raise FileNotFoundError("Cannot retrieve gene information. gene_info_file is required when gctx file has no row metadata.")

# Create gene ID to gene symbol mapping
# Map gctx file row index (gene ID) to gene symbol
if 'pr_gene_symbol' in gene_info_df.columns:
    if 'pr_gene_id' in gene_info_df.columns:
        gene_id_to_symbol = dict(zip(gene_info_df['pr_gene_id'].astype(str), gene_info_df['pr_gene_symbol']))
    else:
        # Use index as gene ID
        gene_id_to_symbol = dict(zip(gene_info_df.index.astype(str), gene_info_df['pr_gene_symbol']))
    logger.info(f"Gene ID to symbol mapping count: {len(gene_id_to_symbol)}")
else:
    raise ValueError("Could not retrieve gene symbol information (pr_gene_symbol).")

# Identify glycogene indices
glycogene_indices = []
glycogene_ids = []

for gene_symbol in glycogenes:
    # Reverse lookup gene ID from gene symbol
    matching_ids = [gid for gid, symbol in gene_id_to_symbol.items() 
                    if str(symbol).upper() == gene_symbol.upper()]
    
    if matching_ids:
        for gid in matching_ids:
            gid_str = str(gid)
            if gid_str in gctoo.data_df.index:
                glycogene_indices.append(gid_str)
                glycogene_ids.append(gid)
                break

logger.info(f"Glycogenes found in data: {len(glycogene_indices)}/{len(glycogenes)}")

# Check genes not found in data
missing_genes = set(glycogenes) - set([gene_id_to_symbol.get(str(gid), '') for gid in glycogene_ids])
if missing_genes:
    logger.warning(f"Number of genes not found in data: {len(missing_genes)}")
    logger.warning(f"Examples: {list(missing_genes)[:10]}")

## Extracting Glycogene Data

Extract glycogene expression data from gctx data.

In [ ]:
# Extract glycogene data
logger.info("Extracting glycogene data...")
glycogene_data = gctoo.data_df.loc[glycogene_indices].copy()

# Convert gene IDs to gene symbols (for column names)
glycogene_data.index = [gene_id_to_symbol.get(str(idx), idx) for idx in glycogene_data.index]

logger.info(f"Extracted data shape: {glycogene_data.shape}")
logger.info(f"Number of genes: {len(glycogene_data)}")
logger.info(f"Number of samples: {len(glycogene_data.columns)}")

# Transpose to wide format (samples as rows, genes as columns)
glycogene_data_wide = glycogene_data.T

logger.info(f"Transposed data shape: {glycogene_data_wide.shape}")
logger.info(f"Number of samples: {len(glycogene_data_wide)}")
logger.info(f"Number of genes: {len(glycogene_data_wide.columns)}")

## Loading Metadata

Load compound information (pert_info) and experiment information (sig_info).

In [ ]:
# Load metadata (from gctx file column metadata or external file)
logger.info("Retrieving metadata...")

# Get experiment information (sig_info)
if hasattr(gctoo, 'col_metadata_df') and gctoo.col_metadata_df is not None:
    logger.info("Retrieving experiment information from gctx file column metadata.")
    sig_info_df = gctoo.col_metadata_df.copy()
    # Use index as sig_id (if not present)
    if 'sig_id' not in sig_info_df.columns:
        sig_info_df['sig_id'] = sig_info_df.index.astype(str)
else:
    if sig_info_file.exists():
        logger.info(f"Loading experiment information file: {sig_info_file}")
        sig_info_df = pd.read_csv(sig_info_file, sep='\t', compression='gzip', low_memory=False)
    else:
        logger.warning("Experiment information not found. Using column names as sig_id.")
        sig_info_df = pd.DataFrame({'sig_id': gctoo.data_df.columns.astype(str)})
        sig_info_df.set_index('sig_id', inplace=True)

logger.info(f"Experiment information shape: {sig_info_df.shape}")
logger.info(f"Columns: {sig_info_df.columns.tolist()}")

# Get compound information (pert_info)
if pert_info_file.exists():
    logger.info(f"Loading compound information file: {pert_info_file}")
    pert_info_df = pd.read_csv(pert_info_file, sep='\t', compression='gzip', low_memory=False)
    logger.info(f"Compound information shape: {pert_info_df.shape}")
    logger.info(f"Columns: {pert_info_df.columns.tolist()}")
else:
    logger.warning("Compound information file not found. Attempting to retrieve from experiment information.")
    # Get pert_id from sig_info for later joining
    pert_info_df = None

# Check key columns
if 'pert_id' in sig_info_df.columns:
    logger.info(f"Unique compounds in experiment info: {sig_info_df['pert_id'].nunique()}")
if pert_info_df is not None and 'pert_id' in pert_info_df.columns:
    logger.info(f"Unique compounds in compound info: {pert_info_df['pert_id'].nunique()}")
if 'sig_id' in sig_info_df.columns:
    logger.info(f"Unique experiments: {sig_info_df['sig_id'].nunique()}")
else:
    logger.info(f"Unique experiments (from index): {len(sig_info_df)}")

## Joining Data

Join glycogene data with metadata.

In [ ]:
# Set sample ID (sig_id) as index
# Use gctx file column names (sample IDs)
glycogene_data_wide.index.name = 'sig_id'
glycogene_data_wide = glycogene_data_wide.reset_index()

logger.info("Joining with experiment information...")
# Join with experiment information (by sig_id)
# Handle both cases: sig_id as index or as column
if 'sig_id' in sig_info_df.columns:
    sig_info_df_merge = sig_info_df.reset_index() if sig_info_df.index.name == 'sig_id' else sig_info_df
else:
    # Use index as sig_id
    sig_info_df_merge = sig_info_df.copy()
    sig_info_df_merge['sig_id'] = sig_info_df_merge.index.astype(str)

df_combined = glycogene_data_wide.merge(
    sig_info_df_merge,
    on='sig_id',
    how='left'
)

logger.info(f"Data shape after join: {df_combined.shape}")

# Join with compound information
if pert_info_df is not None and 'pert_id' in df_combined.columns:
    logger.info("Joining with compound information...")
    df_combined = df_combined.merge(
        pert_info_df,
        on='pert_id',
        how='left',
        suffixes=('', '_pert')
    )
    logger.info(f"Data shape after compound info join: {df_combined.shape}")

# Generate sample ID (use sig_id as sample_id, or create new)
if 'sig_id' in df_combined.columns:
    df_combined['sample_id'] = df_combined['sig_id']
else:
    df_combined['sample_id'] = df_combined.index.astype(str)

logger.info(f"Final data shape: {df_combined.shape}")
logger.info(f"Number of columns: {len(df_combined.columns)}")
logger.info(f"Number of records: {len(df_combined)}")

## Data Formatting and Column Name Cleanup

Format data and clean up column names for Snowflake table.

In [ ]:
# Separate metadata columns and gene columns
metadata_columns = [
    'sample_id', 'sig_id', 'pert_id', 'pert_iname', 'pert_type',
    'cell_id', 'cell_iname', 'pert_idose', 'pert_itime', 'pert_itime_unit',
    'distil_id', 'distil_cc_q75', 'distil_ss', 'distil_nsample',
    'inchi_key', 'canonical_smiles', 'pertname', 'pertid'
]

# Identify existing metadata columns
existing_metadata_cols = [col for col in metadata_columns if col in df_combined.columns]
gene_columns = [col for col in df_combined.columns if col not in existing_metadata_cols]

logger.info(f"Number of metadata columns: {len(existing_metadata_cols)}")
logger.info(f"Number of gene columns: {len(gene_columns)}")

# Standardize column names
# Convert gene column names to uppercase (to match Snowflake naming conventions)
df_final = df_combined.copy()
df_final.columns = [col.upper() if col in gene_columns else col for col in df_final.columns]

# Standardize metadata column names
rename_map = {
    'pert_iname': 'pertname',
    'pert_id': 'pertid',
    'cell_iname': 'cell',
    'pert_idose': 'dose',
    'pert_itime': 'timepoint',
}

for old_col, new_col in rename_map.items():
    if old_col in df_final.columns:
        df_final = df_final.rename(columns={old_col: new_col})

# Check required columns
required_cols = ['sample_id']
for col in required_cols:
    if col not in df_final.columns:
        logger.error(f"Required column '{col}' not found")
        raise ValueError(f"Required column '{col}' not found")

logger.info(f"Final data shape: {df_final.shape}")
logger.info(f"Sample columns: {df_final.columns[:10].tolist()}")

## Snowflake Connection Configuration

Establish connection to Snowflake database.

In [ ]:
def load_private_key() -> bytes:
    """Load private key (DER format)"""
    key_path = os.path.expanduser('~/.ssh/snowflake_rsa_key.pem')
    with open(key_path, "rb") as key_file:
        private_key = serialization.load_pem_private_key(
            key_file.read(),
            password=None,
            backend=default_backend()
        )
    # Encode in DER format (format expected by Snowflake connector)
    private_key_der = private_key.private_bytes(
        encoding=serialization.Encoding.DER,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption()
    )
    return private_key_der


def connect_to_snowflake():
    """Connect to Snowflake"""
    try:
        conn = connect(
            user="KOREEDA",
            account="",
            private_key=load_private_key(),
            warehouse="BIOINFORMATICS_XS",
            database="BIOINFORMATICS",
            schema="LINCS",
            role="ACCOUNTADMIN",
        )
        logger.info("Snowflake connection successful")
        return conn
    except Exception as e:
        logger.error(f"Snowflake connection error: {e}")
        raise

# Connect to Snowflake
conn = connect_to_snowflake()

## Creating Snowflake Table

Create the GLYCO_GENES_WIDE table and upload data.

In [ ]:
cursor = conn.cursor()

# Drop existing table (if exists)
table_name = "GLYCO_GENES_WIDE"
drop_table_query = f"""
DROP TABLE IF EXISTS BIOINFORMATICS.LINCS.{table_name}
"""

logger.info(f"Dropping existing table: {table_name}")
cursor.execute(drop_table_query)
logger.info("Table dropped successfully")

# Generate column definitions for table creation
# Define metadata column types
metadata_column_types = {
    'sample_id': 'VARCHAR(255)',
    'pertid': 'VARCHAR(255)',
    'pertname': 'VARCHAR(255)',
    'cell': 'VARCHAR(100)',
    'dose': 'VARCHAR(100)',
    'timepoint': 'VARCHAR(100)',
    'inchi_key': 'VARCHAR(255)',
    'canonical_smiles': 'VARCHAR(4000)',
    'sig_id': 'VARCHAR(255)',
    'pert_type': 'VARCHAR(100)',
    'cell_id': 'VARCHAR(100)',
}

# Gene columns are all FLOAT type
gene_column_type = 'FLOAT'

# Build CREATE TABLE statement
column_definitions = []

# Metadata columns
for col in existing_metadata_cols:
    if col in metadata_column_types:
        column_definitions.append(f'"{col.upper()}" {metadata_column_types[col]}')
    else:
        column_definitions.append(f'"{col.upper()}" VARCHAR(255)')

# Gene columns
for gene_col in gene_columns:
    column_definitions.append(f'"{gene_col.upper()}" {gene_column_type}')

create_table_query = f"""
CREATE TABLE BIOINFORMATICS.LINCS.{table_name} (
    {', '.join(column_definitions)}
)
"""

logger.info("Creating table...")
logger.info(f"Number of columns: {len(column_definitions)}")
cursor.execute(create_table_query)
logger.info(f"Table created successfully: {table_name}")

## Uploading Data

Upload data to the created table. Batch processing is used due to large data volume.

In [ ]:
# Upload data to Snowflake
# Convert column names to uppercase
df_upload = df_final.copy()
df_upload.columns = [col.upper() for col in df_upload.columns]

# Convert data types (replace NaN with NULL)
df_upload = df_upload.replace({np.nan: None})

# Set batch size (considering memory usage)
batch_size = 10000
total_rows = len(df_upload)
num_batches = (total_rows + batch_size - 1) // batch_size

logger.info(f"Starting data upload")
logger.info(f"Total records: {total_rows:,}")
logger.info(f"Number of batches: {num_batches}")
logger.info(f"Batch size: {batch_size:,}")

# Get column names
columns = df_upload.columns.tolist()
placeholders = ', '.join(['?' for _ in columns])
insert_query = f"""
INSERT INTO BIOINFORMATICS.LINCS.{table_name} ({', '.join([f'"{col}"' for col in columns])})
VALUES ({placeholders})
"""

# Upload in batches
for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, total_rows)
    batch_df = df_upload.iloc[start_idx:end_idx]
    
    # Convert data to list of tuples
    values = [tuple(row) for row in batch_df.values]
    
    # Upload batch
    cursor.executemany(insert_query, values)
    
    if (i + 1) % 10 == 0 or (i + 1) == num_batches:
        logger.info(f"Progress: {i + 1}/{num_batches} batches complete ({end_idx:,}/{total_rows:,} records)")

# Commit
conn.commit()
logger.info("Data upload complete")

## Verifying Data

Verify the uploaded data.

In [ ]:
# Check table record count
count_query = f"SELECT COUNT(*) FROM BIOINFORMATICS.LINCS.{table_name}"
cursor.execute(count_query)
row_count = cursor.fetchone()[0]
logger.info(f"Number of records in table: {row_count:,}")

# Get sample data
sample_query = f"SELECT * FROM BIOINFORMATICS.LINCS.{table_name} LIMIT 5"
sample_df = pd.read_sql(sample_query, conn)
logger.info(f"\nSample data (first 5 rows):")
logger.info(f"Number of columns: {len(sample_df.columns)}")
logger.info(f"\n{sample_df.head()}")

# Statistics
stats_query = f"""
SELECT 
    COUNT(*) as total_records,
    COUNT(DISTINCT PERTID) as unique_compounds,
    COUNT(DISTINCT CELL) as unique_cells,
    COUNT(DISTINCT TIMEPOINT) as unique_timepoints
FROM BIOINFORMATICS.LINCS.{table_name}
"""
stats_df = pd.read_sql(stats_query, conn)
logger.info(f"\nStatistics:")
logger.info(stats_df)

cursor.close()
conn.close()
logger.info("\nProcessing complete")